In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

In [ ]:
PROJECT_PATH = Path.cwd().parent
ARCHIVE_PATH = PROJECT_PATH / "archive"
DATA_PATH = PROJECT_PATH / "data"
MAPPINGS_PATH = PROJECT_PATH / "mappings"

print(PROJECT_PATH)
print(ARCHIVE_PATH)
print(DATA_PATH)
print(MAPPINGS_PATH)

In [ ]:
def inspect_dataset(bank: str, filename: str) -> pd.DataFrame:
    """
    Inspect one raw CSV dataset for a given file name and return it as a data frame.
    
    Displays the first two and last rows, column types, and NaN counts for each CSV file found in the bank's raw data directory.

    Args:
        filename: the name of the file that contains the dataset.

    Returns:
        A data frame.

    Raises:
    """
    df = pd.read_csv(DATA_PATH / "raw" / bank /filename)
    
    display(df.head(2))
    display(df.tail(2))
    print("Types: ")
    display(df.dtypes)
    print("\n")
    print("Nans: ")
    display(df.isna().sum().to_dict())
    print("\n")

    return df
    

DF = inspect_dataset("revolut", "revolut_euro_2020-2026.csv")

In [ ]:
DF[DF.isna().any(axis=1)]

In [ ]:
for c in ["Type", "Product", "Currency", "State"]:
    print(DF[c].unique(), end="\n\n\n")

In [ ]:
def clean_data_revolut(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["State"] != "REVERTED"]
    df = df.drop(columns=["Type", "Product", "Completed Date", "State"])
    df = df.rename(columns={
        "Started Date": "date",
        "Description": "description",
        "Amount": "amount",
        "Fee": "fee",
        "Currency": "currency",
        "Balance": "balance"
    })
    df["date"] = pd.to_datetime(df["date"])
    print(f"New columns of the DataFrame: {list(df.columns)}")
    
    return df

DF = clean_data_revolut(DF)

In [ ]:
def extract_descriptions(df: pd.DataFrame) -> None:
    
    print(f"{len(df['description'].unique())} unique descriptions detected.")
    
    descriptions = dict()
    for d in df["description"].unique():
        descriptions[d] = "blank"

    out_dir = ARCHIVE_PATH / "blank_mappings" / "revolut"
    out_dir.mkdir(parents=True, exist_ok=True)

    with open(out_dir / "revolut_euro_blank_description_mapping.json", "w") as f:
        json.dump(descriptions, f, sort_keys=True, ensure_ascii=False, indent=4)
    
extract_descriptions(DF)       

Upon investigation, it was found that many transactions from a revolut account to another
revolut account is labeled as "Revolut Bank UAB" in the description column. 

The Revolut app is able to provide more information about these transactions.

Thus the upcoming workflow will be:
1. Detect all these transactions in our df.
2. Extract the ones that have a negative in a separate TXT file, and with the help of the revolut app try to       replace the generic "Revolut Bank UAB" with a more useful description. We care only for the negative amounts,    because they signify expenses. The positive one will be labeled as "Payback".
3. After identifying the new descriptions, we need to replace the generic "Revolut Bank UAB" with the new ones inside the data frame.


In [ ]:
DF[
    (DF["description"].str.contains("Revolut", case=False, na=False))
].sort_values("date", ascending=False)

In [ ]:
def extract_negative_revolut_to_revolut(df: pd.DataFrame) -> None:
    entries = []
    
    r2r = df[
        (df["description"].str.contains("Revolut", case=False, na=False)) &
        (df["amount"] < 0)
        ].sort_values("date", ascending=False).reset_index()
    
    out_file = ARCHIVE_PATH / "blank_mappings" / "revolut" / "revolut_euro_new_categories.json"

    with open(out_file, "w", encoding="utf-8") as f:
        for i, row in r2r.iterrows():
            entries.append(
                {
                    "index": i,
                    "date": str(row["date"]),
                    "amount": row["amount"],
                    "new_description": ""
                }
            )
        json.dump(entries, f, ensure_ascii=False, indent=4)

extract_negative_revolut_to_revolut(DF)       

In [ ]:
def replace_negative_p2p_descriptions(df: pd.DataFrame) -> pd.DataFrame:

    new_descriptions = set()
    
    with open(MAPPINGS_PATH / "completed" / "revolut_euro_new_categories.json") as f:
        descriptions = json.load(f)

    print(f"Found {len(descriptions)} descriptions that need replacement.")
    
    for d in descriptions:
        new_descriptions.add(d["new_description"])
        condition = (
            (df["description"].str.contains("Revolut", case=False, na=False)) & # broad match intentional — Revolut Ltd is also P2P, one-time use only
            (df["date"] == pd.to_datetime(d["date"])) &
            (df["amount"] == d["amount"])
        )
        df.loc[condition, "description"] = d["new_description"]

    remaining = df[
        df["description"].str.contains("Revolut", case=False, na=False) & df["amount"] < 0
        ].sort_values("date", ascending=False)

    
    
    if remaining.empty:
        print("All NEGATIVE 'Revolut Bank UAB' & 'Revolut Ltd' entries replaced successfully.")
    else:
        print(f"Warning: f{len(remaining)} unresolved rows remain.")

    return df
    
    
DF = replace_negative_p2p_descriptions(DF)

In [ ]:
print(sorted(new_descriptions))

In [ ]:
def replace_positive_p2p_descriptions(df: pd.DataFrame) -> pd.DataFrame:
    
    condition = (
        (df["description"] == "Revolut Bank UAB") &
        (df["amount"] > 0)
    )


    df.loc[condition, "description"] = "P2P: Payback"

    is_empty = df[df["description"].str.contains("Revolut", case=False, na=False)].empty

    if is_empty:
        print("All POSITIVE and NEGATIVE 'Revolut Bank UAB' & 'Revolut Ltd' entries replaced successfully.")
    return df

DF = replace_positive_p2p_descriptions(DF)

In [ ]:
def create_categories_column(df: pd.DataFrame):
    df = df.sort_values("date").reset_index(drop=True)
    file = MAPPINGS_PATH / "completed" / "revolut_euro.json"
    with open(file) as f:
        mappings = json.load(f)

    df["categories"] = df["description"].map(mappings)

    return df

DF = create_categories_column(DF)

In [ ]:
print(DF["date"].min(), end="\n\n\n")
print(DF["date"].max(), end="\n\n\n")
print(DF.dtypes, end="\n\n\n")
print(DF.isna().sum().to_dict(), end="\n\n\n")
display(DF.head(5))
display(DF.tail(3))

In [ ]:
def create_parquet(df: pd.DataFrame, parquet_name: str):
    out_dir = DATA_PATH / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_parquet(out_dir / parquet_name)

create_parquet(DF, "revolut_euro_2020-02_2026-05.parquet")